# Elastic Inference Service: One API, Every Model

Explore the inference endpoints on your cluster, compare embedding models, and use LLMs — all through the same Inference API.

**Time:** ~15–20 minutes

## 1. Setup & Connect

In [ ]:
!pip install elasticsearch requests -q

In [ ]:
import os
from elasticsearch import Elasticsearch

ELASTICSEARCH_URL = os.environ.get("ELASTICSEARCH_URL") or input("Elasticsearch URL: ")
ELASTIC_API_KEY = os.environ.get("ELASTIC_API_KEY") or input("Elastic API Key: ")

es = Elasticsearch(ELASTICSEARCH_URL, api_key=ELASTIC_API_KEY)
print(f"Connected to Elasticsearch {es.info()['version']['number']}")

## 2. What's Already on Your Cluster

Every Serverless and Cloud Hosted cluster comes with **pre-configured inference endpoints**. These start with a dot (`.`) and are ready to use immediately — no setup needed.

In [ ]:
endpoints = es.inference.get(inference_id="_all")

from collections import defaultdict
by_type = defaultdict(list)

for ep in endpoints["endpoints"]:
    task = ep.get("task_type", "unknown")
    eid = ep["inference_id"]
    service = ep.get("service", "—")
    model = ep.get("service_settings", {}).get("model_id", "—")
    by_type[task].append({"id": eid, "service": service, "model": model})

for task_type, eps in sorted(by_type.items()):
    print(f"\n{'='*60}")
    print(f"  {task_type.upper()} ({len(eps)} endpoints)")
    print(f"{'='*60}")
    for ep in eps:
        prefix = "\N{PUSHPIN}" if ep["id"].startswith(".") else "  "
        print(f"  {prefix} {ep['id']}")
        print(f"     service: {ep['service']}  model: {ep['model']}")

📌 = pre-configured (dot-prefix). These exist out of the box. Everything else was created by a user or application.

## 3. The Three Modes

All three modes use the same Inference API. The only difference is where the model runs.

### Mode 1: Pre-configured (zero setup)

Just reference the dot-prefixed ID. The endpoint already exists.

In [ ]:
# Mode 1: Use a pre-configured endpoint — no PUT needed
result = es.inference.inference(
    inference_id=".jina-embeddings-v5-text-small",
    body={"input": ["What is the EU AI Act?"]},
)
emb = result["text_embedding"][0]["embedding"]
print(f"Mode 1 (pre-configured): {len(emb)}-dimensional vector")
print(f"First 5 values: {[round(v, 4) for v in emb[:5]]}")

### Mode 2: Custom EIS endpoint

Create your own endpoint pointing to an EIS-managed model. Still runs on Elastic's infrastructure.

In [ ]:
# Mode 2: Create a custom endpoint, still on EIS
try:
    es.inference.put(
        inference_id="my-custom-elser",
        task_type="sparse_embedding",
        body={
            "service": "elastic",  # "elastic" = runs on EIS infrastructure
            "service_settings": {
                "model_id": "elser_model_2",
            },
        },
    )
    print("Custom ELSER endpoint created (runs on EIS)")
except Exception as e:
    if "already exists" in str(e).lower() or "resource_already_exists" in str(e).lower():
        print("Custom ELSER endpoint already exists")
    else:
        raise

### Mode 3: BYOK (Bring Your Own Key)

External provider, your API key, same Inference API shape.

In [ ]:
# Mode 3: BYOK — uncomment to try with your own OpenAI key
# OPENAI_KEY = input("OpenAI API Key: ")
# es.inference.put(
#     inference_id="my-openai-embeddings",
#     task_type="text_embedding",
#     body={
#         "service": "openai",           # ← provider name, not "elastic"
#         "service_settings": {
#             "model_id": "text-embedding-3-small",
#             "api_key": OPENAI_KEY,      # ← your key
#         },
#     },
# )

print('Mode 3: Same API shape. Only difference:')
print('  "service": "openai"  instead of  "service": "elastic"')

## 4. Compare Embedding Models

Same text, different models. Dense vs sparse, different dimensions.

In [ ]:
test_text = "Facial recognition systems used by law enforcement"

models = {
    ".jina-embeddings-v5-text-small": "Jina v5 (dense)",
    ".elser-2-elastic": "ELSER v2 (sparse)",
}

print(f'Text: "{test_text}"\n')
for endpoint_id, label in models.items():
    try:
        result = es.inference.inference(
            inference_id=endpoint_id,
            body={"input": [test_text]},
        )
        if "text_embedding" in result:
            emb = result["text_embedding"][0]["embedding"]
            print(f"  {label}: {len(emb)}-dimensional dense vector")
            print(f"    First 5 values: {[round(v, 4) for v in emb[:5]]}")
        elif "sparse_embedding" in result:
            emb = result["sparse_embedding"][0]
            # Sparse response has an "embedding" key with {token: weight} pairs
            tokens = emb.get("embedding", emb)
            if isinstance(tokens, dict):
                # Filter to numeric values only, sort by weight
                numeric = {k: v for k, v in tokens.items() if isinstance(v, (int, float))}
                top = sorted(numeric.items(), key=lambda x: x[1], reverse=True)[:5]
                print(f"  {label}: sparse vector, {len(numeric)} active terms")
                print(f"    Top terms: {top}")
            else:
                print(f"  {label}: sparse embedding returned")
                print(f"    Keys: {list(emb.keys()) if isinstance(emb, dict) else type(emb)}")
        print()
    except Exception as e:
        print(f"  {label}: {e}\n")

## 5. Chat/Completion via EIS

EIS also provides managed LLMs — the same models that power AI Playground, Agent Builder, and AI Assistant in Kibana.

In [ ]:
# Find available chat endpoints
chat_endpoints = [
    ep for ep in endpoints["endpoints"]
    if ep.get("task_type") in ("completion", "chat_completion")
]

if chat_endpoints:
    chat_id = chat_endpoints[0]["inference_id"]
    model = chat_endpoints[0].get("service_settings", {}).get("model_id", "unknown")
    print(f"Using: {chat_id} ({model})\n")

    # Chat completion requires streaming on EIS — use requests directly
    import requests, json

    es_url = ELASTICSEARCH_URL.rstrip("/")
    resp = requests.post(
        f"{es_url}/_inference/chat_completion/{chat_id}/_stream",
        headers={
            "Content-Type": "application/json",
            "Authorization": f"ApiKey {ELASTIC_API_KEY}",
        },
        json={
            "messages": [
                {"role": "user", "content": "In one paragraph, what is the EU AI Act and why should companies care?"}
            ]
        },
        stream=True,
    )

    full_text = ""
    for line in resp.iter_lines():
        if not line:
            continue
        line = line.decode("utf-8")
        if line == "data: [DONE]":
            break
        if line.startswith("data: "):
            try:
                chunk = json.loads(line[6:])
                delta = chunk.get("choices", [{}])[0].get("delta", {}).get("content", "")
                full_text += delta
            except (json.JSONDecodeError, IndexError, KeyError):
                pass

    print(full_text or "No response received")
else:
    print("No chat/completion endpoints found.")
    print("Available on Serverless and Cloud Hosted deployments.")

## 6. What to Tell Customers

**Billing:** Per million tokens. Embeddings = input tokens only. LLMs = input + output tokens.
Check usage: Cloud Console → Billing → Usage → "Inference" dimension.

**Rate limits:** Jina Embeddings 6K req/min • Rerankers 600 req/min • LLMs 2K req/min.
Limits can be raised — work with the Elastic inference team with a business justification.

**Data:** 0-day retention. Not used for model training.

**Regions:** AWS us-east-1 • GCP us-east4 (new) — Europe and Asia regions coming soon.

**Pricing:** [Cloud pricing table](https://cloud.elastic.co/cloud-pricing-table?productType=serverless)